# SQL Advanced Practice Exercises

**Estimated time:** ~8 hours (part of the ~14 hour sql-advanced level, alongside `sql-advanced-guide.ipynb`).

These follow `sql-advanced-guide.ipynb` in order, and every heading names the **guide section** it practises.
Numbering starts at 2 because guide section 1 is the setup, which the cell below covers.

## How to Use This Notebook

- **Run the setup cell first.** It builds the shop database in memory, generates the 200,000-row `events` table,
  and defines four helpers: `q(sql)`, `run(sql)`, `plan(sql)` and `timed(sql)`.
- Write your answer where the comment says to, then run the cell with **Shift + Enter**.
- Every cell ends with `assert` checks. You are right when it prints `OK` with no `AssertionError`.
- **The performance exercises check the query plan, not the clock.** Timings vary between machines and runs; a
  plan does not. When an exercise asks you to make something faster, the check is that the plan changed in the
  right way.
- Several exercises create indexes or tables. They all start by dropping what they create, so every cell can be
  run twice.
- If a cell leaves the database in a state you do not like, re-run the setup cell. It rebuilds everything.
- Revenue means `quantity * unit_price * (1 - discount)`, and cancelled orders are excluded unless an exercise
  says otherwise.

## Setup — Run This First

Builds the shop tables, generates `events`, and defines `q`, `run`, `plan` and `timed`. Generating 200,000 rows
takes a moment. Run it once; run it again after any kernel restart.

In [1]:
print("type")

type


In [2]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")


# ---- a bigger table, so that timings and query plans mean something ----
run("""
DROP TABLE IF EXISTS events;

CREATE TABLE events (
    event_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    event_type  TEXT    NOT NULL,
    event_date  TEXT    NOT NULL,
    amount      REAL    NOT NULL
);

INSERT INTO events (event_id, customer_id, event_type, event_date, amount)
WITH RECURSIVE seq(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 200000
)
SELECT n,
       1 + (n * 7919) % 60,
       CASE WHEN (n * 31) % 100 < 55 THEN 'view'
            WHEN (n * 31) % 100 < 80 THEN 'search'
            WHEN (n * 31) % 100 < 93 THEN 'cart'
            ELSE 'checkout' END,
       date('2023-01-01', '+' || (n % 730) || ' days'),
       ROUND(((n * 37) % 5000) / 10.0, 2)
FROM seq;
""")

print("events      ", q("SELECT COUNT(*) AS n FROM events")["n"][0], "rows")


import time


def plan(sql):
    """The query plan as a list of readable lines."""
    return q("EXPLAIN QUERY PLAN " + sql)["detail"].tolist()


def timed(sql, runs=5):
    """Median wall time of a query in milliseconds, after a warm-up run."""
    q(sql)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        q(sql)
        times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[len(times) // 2]


def drop_practice_indexes():
    """Drop every index these exercises created, so each one starts from a clean slate."""
    names = q("SELECT name FROM sqlite_master WHERE type = 'index' AND name LIKE 'ex_%'")["name"]
    for name in names:
        run(f"DROP INDEX {name}")

categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows
events       200000 rows


## Exercise 2: Reading a Plan  *(guide section 3)*

No SQL to write — read four plans and record what you see. Set each variable to a **string**, one of
`'scan'`, `'search'` or `'sort'`:

- `p1` — the plan for `SELECT * FROM events WHERE amount > 400`
- `p2` — the plan for `SELECT * FROM events WHERE event_id = 500`
- `p3` — whether `SELECT * FROM events ORDER BY amount DESC LIMIT 5` needs a run-time sort (`'sort'` if the plan
  mentions a temporary B-tree, otherwise `'scan'`)
- `p4` — the plan for `SELECT * FROM orders WHERE order_id = 12`

Use `plan(sql)` to look at each one first, then answer. Nothing here has an index on it yet apart from the
primary keys.

In [ ]:
# look at them first
for sql in ["SELECT * FROM events WHERE amount > 400",
            "SELECT * FROM events WHERE event_id = 500",
            "SELECT * FROM events ORDER BY amount DESC LIMIT 5",
            "SELECT * FROM orders WHERE order_id = 12"]:
    print(plan(sql), " <- ", sql)

p1 = ...
p2 = ...
p3 = ...
p4 = ...

assert p1 == "scan", "no index on amount, so every row is read"
assert p2 == "search", "event_id is an INTEGER PRIMARY KEY -- a direct rowid lookup"
assert p3 == "sort", "look for USE TEMP B-TREE FOR ORDER BY"
assert p4 == "search", "same reason as p2"
print("OK")

## Exercise 3: Statistics  *(guide section 2)*

Refresh the planner's statistics and read them back.

1. Run `ANALYZE` (with `run`, not `q` — it returns no rows).
2. Set `stats` to the result of
   `SELECT tbl, idx, stat FROM sqlite_stat1 WHERE tbl = 'events'`.
3. Set `events_rows` to the number of rows `sqlite_stat1` reports for `events` — the **first** whitespace-separated
   number in the `stat` string, as an `int`.

`stat` is text like `'200000 50000'`: total rows, then average rows per distinct value of the indexed column.

In [ ]:
# Your code here

stats = ...
events_rows = ...

assert list(stats.columns) == ["tbl", "idx", "stat"]
assert len(stats) >= 1, "run ANALYZE first, or sqlite_stat1 will be empty"
assert events_rows == 200000
print("OK")

## Exercise 4: Your First Index  *(guide section 4)*

Make a lookup fast and prove it.

The query is `SELECT COUNT(*) AS n FROM events WHERE customer_id = 42`.

1. Drop any index called `ex_events_customer`, then record `before = plan(lookup)`.
2. Create an index called **`ex_events_customer`** on `events(customer_id)`.
3. Record `after = plan(lookup)`.

The checks require the plan to change from a scan to a search that names your index.

In [3]:
lookup = "SELECT COUNT(*) AS n FROM events WHERE customer_id = 42"

run(" drop index if exists ex_events_customer")

drop_practice_indexes()
before = plan(lookup)


run(" create index ex_events_customer on events(customer_id)")
after = plan(lookup)
print("before:", before)
print("after :", after)

assert any("SCAN" in line for line in before), "with no index this has to read every row"
assert any("SEARCH" in line for line in after), "the plan should now be a SEARCH"
assert any("ex_events_customer" in line for line in after), "name the index ex_events_customer"
assert q(lookup)["n"][0] == q("SELECT COUNT(*) AS n FROM events WHERE customer_id = 42")["n"][0]
print("OK")

before: ['SCAN events']
after : ['SEARCH events USING COVERING INDEX ex_events_customer (customer_id=?)']
OK


## Exercise 5: Composite Column Order  *(guide section 5)*

Two candidate indexes for the same pair of columns. Find out which one serves which query.

The two queries are already written below. Create the two indexes **one at a time** — the second only after
dropping the first — and record the plans:

- `ab_both` / `ab_type` — plans for the two queries with an index on `(event_type, event_date)`, named
  `ex_ev_type_date`
- `ba_both` / `ba_type` — plans for the same two queries with an index on `(event_date, event_type)`, named
  `ex_ev_date_type`

Then set `answer` to the name of the index that lets `q_type` — the one filtering on `event_type` alone — use a
`SEARCH`.

In [4]:
q_both = ("SELECT COUNT(*) AS n FROM events "
          "WHERE event_type = 'cart' AND event_date >= '2024-06-01'")
q_type = "SELECT COUNT(*) AS n FROM events WHERE event_type = 'cart'"

drop_practice_indexes()

# create ex_ev_type_date, then record ab_both and ab_type
run("CREATE INDEX ex_ev_type_date ON events(event_type, event_date)")


ab_both = plan(q_both)
ab_type = plan(q_type)

# drop it, create ex_ev_date_type, then record ba_both and ba_type
run("DROP INDEX ex_ev_type_date")
run("CREATE INDEX ex_ev_date_type ON events(event_date, event_type)")

ba_both = plan(q_both)
ba_type = plan(q_type)
answer = "ex_ev_type_date"

assert any("SEARCH" in line for line in ab_both)
assert any("SEARCH" in line for line in ab_type), "event_type is the leading column, so this can search"
assert any("SEARCH" in line for line in ba_both)
assert not any("SEARCH" in line for line in ba_type), "event_type is the second column here, so the leftmost prefix rule blocks a search"
assert answer == "ex_ev_type_date"
print("OK")

OK


## Exercise 6: A Covering Index  *(guide section 6)*

Turn a `SEARCH` into a `SEARCH ... USING COVERING INDEX`.

The report is

```sql
SELECT customer_id, COUNT(*) AS events, SUM(amount) AS total
FROM events
WHERE event_type = 'search'
GROUP BY customer_id
```

1. Drop `ex_ev_cover`, create an index called **`ex_ev_cover`** on `events(event_type)` only, and record
   `narrow = plan(report)`.
2. Drop it and create `ex_ev_cover` again with **every column the query touches**, in an order that puts the
   filter first, then the grouping column, then the measure. Record `covering = plan(report)`.

The second plan should say `COVERING INDEX`, and the temporary B-tree for the `GROUP BY` should disappear as
well — the index already delivers the rows in customer order.

In [5]:
report = """
SELECT customer_id, COUNT(*) AS events, SUM(amount) AS total
FROM events
WHERE event_type = 'search'
GROUP BY customer_id
"""

drop_practice_indexes()
# create the narrow index, then:
run("CREATE INDEX ex_ev_cover ON events(event_type)")
narrow = plan(report)

drop_practice_indexes()
run("CREATE INDEX ex_ev_cover ON events(event_type, customer_id, amount)")
covering = plan(report)

print("narrow  :", narrow)
print("covering:", covering)

assert any("SEARCH" in line for line in narrow)
assert not any("COVERING INDEX" in line for line in narrow), "one column cannot cover this query"
assert any("COVERING INDEX" in line for line in covering), "the index must hold every column used"
assert not any("TEMP B-TREE" in line for line in covering), "put customer_id before amount and the GROUP BY needs no sort"
assert len(q(report)) == 60
print("OK")

narrow  : ['SEARCH events USING INDEX ex_ev_cover (event_type=?)', 'USE TEMP B-TREE FOR GROUP BY']
covering: ['SEARCH events USING COVERING INDEX ex_ev_cover (event_type=?)']
OK


## Exercise 7: Making a Filter SARGable  *(guide section 7)*

This query is correct and cannot use an index:

```sql
SELECT COUNT(*) AS n FROM events WHERE strftime('%Y-%m', event_date) = '2024-03'
```

1. Create an index called **`ex_ev_date`** on `events(event_date)`.
2. Set `slow_plan` to the plan of the query above.
3. Set `fast_sql` to a rewrite that returns **the same count** and can use the index — a half-open range on
   `event_date` itself, with no function wrapped around the column.
4. Set `fast_plan` to its plan, and `same` to whether the two queries return the same number.

In [8]:
slow_sql = "SELECT COUNT(*) AS n FROM events WHERE strftime('%Y-%m', event_date) = '2024-03'"

drop_practice_indexes()
# Your CREATE INDEX here
run("DROP INDEX IF EXISTS ex_ev_date")
run("CREATE INDEX ex_ev_date ON events(event_date)")

slow_plan = plan(slow_sql)
fast_sql = """
SELECT COUNT(*) AS n
FROM events
WHERE event_date >= '2024-03-01'
  AND event_date < '2024-04-01'
"""
fast_plan = plan(fast_sql)

same = q(fast_sql)["n"][0] == q(slow_sql)["n"][0]
print("slow plan:", slow_plan)
print("fast plan:", fast_plan)
print("same:", same)

assert not any("SEARCH" in line for line in slow_plan), "a function around the column blocks the index"
assert any("SEARCH" in line for line in fast_plan), "the rewrite should SEARCH the index"
assert "strftime" not in fast_sql, "the rewrite must not put a function on event_date"
assert q(fast_sql)["n"][0] == q(slow_sql)["n"][0]
print("OK")

slow plan: ['SCAN events USING COVERING INDEX ex_ev_date']
fast plan: ['SEARCH events USING COVERING INDEX ex_ev_date (event_date>? AND event_date<?)']
same: True
OK


## Exercise 8: Measuring Honestly  *(guide section 8)*

Compare two ways of counting the same thing, properly.

- `a` — the median time of `SELECT COUNT(*) AS n FROM events WHERE customer_id = 7 AND event_type = 'cart'`
  with **no** supporting index (drop `ex_ev_ct` first).
- `b` — the median time of the same query with an index called **`ex_ev_ct`** on
  `events(customer_id, event_type)`.

Use the `timed()` helper, which already warms up and takes a median. Then set `faster` to `True` if the plan
changed from a scan to a search — the plan, not the clock, is the evidence.

In [54]:
lookup = "SELECT COUNT(*) AS n FROM events WHERE customer_id = 7 AND event_type = 'cart'"

drop_practice_indexes()
a = timed(lookup)
plan_a = plan(lookup)

run("""
CREATE INDEX ex_ev_ct
ON events(customer_id, event_type)
""")

b = timed(lookup)
plan_b = plan(lookup)

faster = True
print(f"no index {a:6.2f} ms | index {b:6.2f} ms")

assert isinstance(a, float) and isinstance(b, float), "use timed(), which returns a float"
assert any("SCAN" in line for line in plan_a)
assert any("SEARCH" in line for line in plan_b) and any("ex_ev_ct" in line for line in plan_b)
assert faster is True
print("OK")

no index   6.40 ms | index   0.25 ms
OK


## Exercise 9: Recursive Hierarchy  *(guide section 9)*

Walk the `employees` tree from the founder down.

Columns:

- `employee_id`
- `name`
- `depth` — 0 for the founder, 1 for their direct reports, and so on
- `path` — the chain of names from the founder, joined with `' > '`

Order by `path`.

Anchor on the row whose `manager_id` is `NULL`, then join `employees` back to the CTE on
`manager_id = employee_id`.

In [14]:
sql = """
WITH RECURSIVE org AS (
    SELECT employee_id,
           name,
           role,
           0                       AS depth,
           name                    AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT e.employee_id,
           e.name,
           e.role,
           o.depth + 1,
           o.path || ' > ' || e.name
    FROM employees e
    JOIN org o ON o.employee_id = e.manager_id
)
SELECT employee_id,
        name ,
       depth,
       path
FROM org
ORDER BY path
"""

out = q(sql)
print(out)
expected = [
    [1, "Radhika Menon", 0, "Radhika Menon"],
    [4, "Imran Sheikh", 1, "Radhika Menon > Imran Sheikh"],
    [12, "Meera Iyer", 2, "Radhika Menon > Imran Sheikh > Meera Iyer"],
    # ...
]

for i, (actual, exp) in enumerate(zip(out.values.tolist(), expected)):
    if actual != exp:
        print("Mismatch at row:", i)
        print("Actual:  ", actual)
        print("Expected:", exp)

assert list(out.columns) == ["employee_id", "name", "depth", "path"]
assert len(out) == 15
assert out.round(2).values.tolist() == [
    [1, "Radhika Menon", 0, "Radhika Menon"],
    [4, "Imran Sheikh", 1, "Radhika Menon > Imran Sheikh"],
    [12, "Meera Iyer", 2, "Radhika Menon > Imran Sheikh > Meera Iyer"],
    [11, "Sanjay Gupta", 2, "Radhika Menon > Imran Sheikh > Sanjay Gupta"],
    [13, "Tarun Das", 2, "Radhika Menon > Imran Sheikh > Tarun Das"],
    [3, "Sunita Rao", 1, "Radhika Menon > Sunita Rao"],
    [10, "Farah Qureshi", 2, "Radhika Menon > Sunita Rao > Farah Qureshi"],
    [9, "Nikhil Verma", 2, "Radhika Menon > Sunita Rao > Nikhil Verma"],
    [15, "Omar Farooq", 2, "Radhika Menon > Sunita Rao > Omar Farooq"],
    [8, "Priya Balan", 2, "Radhika Menon > Sunita Rao > Priya Balan"],
    [2, "Vikram Nair", 1, "Radhika Menon > Vikram Nair"],
    [5, "Arjun Pillai", 2, "Radhika Menon > Vikram Nair > Arjun Pillai"],
    [7, "Devendra Joshi", 2, "Radhika Menon > Vikram Nair > Devendra Joshi"],
    [6, "Kavya Krishnan", 2, "Radhika Menon > Vikram Nair > Kavya Krishnan"],
    [14, "Lakshmi Suresh", 2, "Radhika Menon > Vikram Nair > Lakshmi Suresh"],
]
print("OK")


    employee_id            name  depth                                          path
0             1   Radhika Menon      0                                 Radhika Menon
1             4    Imran Sheikh      1                  Radhika Menon > Imran Sheikh
2            12      Meera Iyer      2     Radhika Menon > Imran Sheikh > Meera Iyer
3            11    Sanjay Gupta      2   Radhika Menon > Imran Sheikh > Sanjay Gupta
4            13       Tarun Das      2      Radhika Menon > Imran Sheikh > Tarun Das
5             3      Sunita Rao      1                    Radhika Menon > Sunita Rao
6            10   Farah Qureshi      2    Radhika Menon > Sunita Rao > Farah Qureshi
7             9    Nikhil Verma      2     Radhika Menon > Sunita Rao > Nikhil Verma
8            15     Omar Farooq      2      Radhika Menon > Sunita Rao > Omar Farooq
9             8     Priya Balan      2      Radhika Menon > Sunita Rao > Priya Balan
10            2     Vikram Nair      1                   Radhika 

## Exercise 10: A Date Spine  *(guide section 10)*

Daily order counts for **November 2024**, with a row for every day of the month whether or not anything was
ordered.

Columns `day` (`'YYYY-MM-DD'`), `orders` and `revenue` (rounded to 2). Cancelled orders excluded. Order by
`day`.

Generate the 30 days with a recursive CTE, `LEFT JOIN` the real data onto that spine, and `COALESCE` the empty
days to zero. The result must have exactly 30 rows — several of them zeros.

In [19]:
sql = """
with recursive tble(monthd) as (
select '2024-11-01'

union all

select date(monthd,'+1 day') 
from tble
where monthd < '2024-11-30'
),
sales AS (
    SELECT o.order_date,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue,
           COUNT(DISTINCT o.order_id)                        AS orders
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.order_date
)
SELECT m.monthd as day,
       COALESCE(s.orders, 0)                AS orders,
       ROUND(COALESCE(s.revenue, 0), 2)     AS revenue
FROM tble m
LEFT JOIN sales s ON s.order_date=m.monthd
ORDER BY monthd
"""

out = q(sql)
print(out)

assert list(out.columns) == ["day", "orders", "revenue"]
assert len(out) == 30
assert out.round(2).values.tolist() == [
    ["2024-11-01", 0, 0],
    ["2024-11-02", 0, 0],
    ["2024-11-03", 0, 0],
    ["2024-11-04", 0, 0],
    ["2024-11-05", 0, 0],
    ["2024-11-06", 2, 85000],
    ["2024-11-07", 1, 16502.5],
    ["2024-11-08", 1, 432800],
    ["2024-11-09", 2, 307477.5],
    ["2024-11-10", 2, 117892.5],
    ["2024-11-11", 1, 56000],
    ["2024-11-12", 0, 0],
    ["2024-11-13", 1, 35960],
    ["2024-11-14", 2, 113650],
    ["2024-11-15", 0, 0],
    ["2024-11-16", 1, 46310],
    ["2024-11-17", 1, 139550],
    ["2024-11-18", 1, 240655],
    ["2024-11-19", 2, 281400],
    ["2024-11-20", 1, 153000],
    ["2024-11-21", 0, 0],
    ["2024-11-22", 1, 102600],
    ["2024-11-23", 1, 105800],
    ["2024-11-24", 0, 0],
    ["2024-11-25", 0, 0],
    ["2024-11-26", 2, 57000],
    ["2024-11-27", 1, 37350],
    ["2024-11-28", 0, 0],
    ["2024-11-29", 0, 0],
    ["2024-11-30", 0, 0],
]
print("OK")

           day  orders  revenue
0   2024-11-01       0      0.0
1   2024-11-02       0      0.0
2   2024-11-03       0      0.0
3   2024-11-04       0      0.0
4   2024-11-05       0      0.0
..         ...     ...      ...
25  2024-11-26       2  57000.0
26  2024-11-27       1  37350.0
27  2024-11-28       0      0.0
28  2024-11-29       0      0.0
29  2024-11-30       0      0.0

[30 rows x 3 columns]
OK


## Exercise 11: ROWS versus RANGE  *(guide section 11)*

Show the two frame types disagreeing on tied values.

Build a five-row list inline with `SELECT ... UNION ALL` — `('a', 5), ('b', 15), ('c', 15), ('d', 15), ('e', 25)`
— with columns `label` and `score`. Then add:

- `by_rows` — `SUM(score) OVER (ORDER BY score ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`
- `by_range` — the same with `RANGE` instead of `ROWS`
- `gap` — `by_range - by_rows`

Order by `score`, then `label`.

The three tied rows should show three different `by_rows` values and one shared `by_range` value.

In [23]:
sql = """
with cte as (select 'a' as label , 5 as value
union all 
select 'b' , 15 
union all 
select 'c' , 15
union all 
select 'd' , 15
union all
select 'e' , 25),
mid as (select label , value as score , sum(value) over (order by value rows between unbounded preceding and current row) as by_rows,
sum(value) over(order by value range between unbounded preceding and current row) as by_range
from cte)
select * , by_range-by_rows as gap
from mid
order by score , label
"""

out = q(sql)
print(out)
assert list(out.columns) == ["label", "score", "by_rows", "by_range", "gap"]
assert len(out) == 5
assert out.round(2).values.tolist() == [
    ["a", 5, 5, 5, 0],
    ["b", 15, 20, 50, 30],
    ["c", 15, 35, 50, 15],
    ["d", 15, 50, 50, 0],
    ["e", 25, 75, 75, 0],
]
print("OK")

  label  score  by_rows  by_range  gap
0     a      5        5         5    0
1     b     15       20        50   30
2     c     15       35        50   15
3     d     15       50        50    0
4     e     25       75        75    0
OK


## Exercise 12: Gaps and Islands  *(guide section 12)*

Find the **streaks of consecutive days on which at least one order was placed**, across the whole of 2024.

Columns `start_date`, `end_date`, `days`. Keep only streaks of **3 days or more**. Order by `days` descending,
then `start_date`.

The trick from the guide: number the distinct active dates with `ROW_NUMBER() OVER (ORDER BY d)`, subtract that
many days from the date, and group by the result. Consecutive dates all produce the same key.

In [35]:
sql = """
WITH cte AS (
    SELECT DISTINCT order_date
    FROM orders
    WHERE strftime('%Y', order_date) = '2024'
),
numbered AS (
    SELECT
        order_date,
        ROW_NUMBER() OVER (ORDER BY order_date) AS rn
    FROM cte
)
 , mid as (SELECT
    order_date,
    rn,
    date(order_date, '-' || rn || ' days') AS island
FROM numbered)
select min(order_date) as start_date , max(order_date) as end_date , count(*) as days
from mid
group by island
having count(*)>=3
order by count(*) desc , start_date
"""

out = q(sql)
print(out)

assert list(out.columns) == ["start_date", "end_date", "days"]
assert len(out) == 17
assert out.round(2).values.tolist() == [
    ["2024-09-16", "2024-09-22", 7],
    ["2024-07-24", "2024-07-29", 6],
    ["2024-11-06", "2024-11-11", 6],
    ["2024-11-16", "2024-11-20", 5],
    ["2024-08-08", "2024-08-11", 4],
    ["2024-08-20", "2024-08-23", 4],
    ["2024-09-01", "2024-09-04", 4],
    ["2024-01-30", "2024-02-01", 3],
    ["2024-03-11", "2024-03-13", 3],
    ["2024-05-07", "2024-05-09", 3],
    ["2024-07-16", "2024-07-18", 3],
    ["2024-08-28", "2024-08-30", 3],
    ["2024-10-10", "2024-10-12", 3],
    ["2024-12-03", "2024-12-05", 3],
    ["2024-12-09", "2024-12-11", 3],
    ["2024-12-23", "2024-12-25", 3],
    ["2024-12-27", "2024-12-29", 3],
]
print("OK")

    start_date    end_date  days
0   2024-09-16  2024-09-22     7
1   2024-07-24  2024-07-29     6
2   2024-11-06  2024-11-11     6
3   2024-11-16  2024-11-20     5
4   2024-08-08  2024-08-11     4
5   2024-08-20  2024-08-23     4
6   2024-09-01  2024-09-04     4
7   2024-01-30  2024-02-01     3
8   2024-03-11  2024-03-13     3
9   2024-05-07  2024-05-09     3
10  2024-07-16  2024-07-18     3
11  2024-08-28  2024-08-30     3
12  2024-10-10  2024-10-12     3
13  2024-12-03  2024-12-05     3
14  2024-12-09  2024-12-11     3
15  2024-12-23  2024-12-25     3
16  2024-12-27  2024-12-29     3
OK


## Exercise 13: Transactions  *(guide section 13)*

Prove that a rollback undoes everything.

A table `ledger` is created for you with two accounts holding 1000 each.

1. Set `before` to the total balance.
2. Open a transaction with `con.execute("BEGIN")`, move 400 from account 1 to account 2 with two `UPDATE`
   statements, and set `during` to the total **without committing**.
3. Roll it back with `con.rollback()` and set `after` to the total.
4. Then do the same transfer again and **commit** it this time. Set `committed` to account 2's balance.

Use `con.execute(...)` directly rather than `run(...)` — `run` uses `executescript`, which commits as it goes
and would defeat the whole exercise.

In [55]:
run("""
DROP TABLE IF EXISTS ledger;
CREATE TABLE ledger (account_id INTEGER PRIMARY KEY, balance REAL NOT NULL CHECK (balance >= 0));
INSERT INTO ledger VALUES (1, 1000), (2, 1000);
""")

def total():
    return float(q("SELECT SUM(balance) AS t FROM ledger")["t"][0])

before = total()
con.execute("BEGIN")

con.execute("""
    UPDATE ledger
    SET balance = balance - 400
    WHERE account_id = 1
""")

con.execute("""
    UPDATE ledger
    SET balance = balance + 400
    WHERE account_id = 2
""")
# your transaction, then a rollback

during = total()

con.rollback()

after = total()


# now do it again and commit

con.execute("BEGIN")

con.execute("""
    UPDATE ledger
    SET balance = balance - 400
    WHERE account_id = 1
""")

con.execute("""
    UPDATE ledger
    SET balance = balance + 400
    WHERE account_id = 2
""")

con.commit()

committed = float(
    q("""
        SELECT balance
        FROM ledger
        WHERE account_id = 2
    """)["balance"][0]
)

assert before == 2000.0
assert during == 2000.0, "a transfer moves money between rows; the total never changes"
assert after == 2000.0
assert float(q("SELECT balance AS b FROM ledger WHERE account_id = 1")["b"][0]) == 600.0, "account 1 should be down 400 after the committed transfer"
assert committed == 1400.0
print("OK")

OK


## Exercise 14: Constraints and Generated Columns  *(guide section 14)*

Push the rules into the schema.

Create a table `booking` with:

- `booking_id INTEGER PRIMARY KEY`
- `customer_id INTEGER NOT NULL` referencing `customers(customer_id)`
- `product_id INTEGER NOT NULL` referencing `products(product_id)`
- `quantity INTEGER NOT NULL` with a `CHECK` that it is greater than zero
- `unit_price REAL NOT NULL` with a `CHECK` that it is not negative
- `total REAL` **generated always as** `quantity * unit_price`, `STORED`
- a `UNIQUE` constraint on the pair `(customer_id, product_id)`

Then insert `(1, 5, 2, 100.0)` and `(1, 6, 1, 250.0)` as `(customer_id, product_id, quantity, unit_price)`.

The checks then try five bad inserts and require every one to be refused.

In [37]:
run("""
CREATE TABLE booking (
    booking_id INTEGER PRIMARY KEY,

    customer_id INTEGER NOT NULL
        REFERENCES customers(customer_id),

    product_id INTEGER NOT NULL
        REFERENCES products(product_id),

    quantity INTEGER NOT NULL
        CHECK (quantity > 0),

    unit_price REAL NOT NULL
        CHECK (unit_price >= 0),

    total REAL
        GENERATED ALWAYS AS (quantity * unit_price) STORED,

    UNIQUE (customer_id, product_id)
);
INSERT INTO booking
(customer_id, product_id, quantity, unit_price)
VALUES
(1, 5, 2, 100.0),
(1, 6, 1, 250.0);
""")

rows = q("SELECT * FROM booking ORDER BY booking_id")

def rejected(sql):
    """True when the database refuses the statement."""
    try:
        run(sql)
        return False
    except Exception:
        return True

assert list(rows.columns) == ["booking_id", "customer_id", "product_id", "quantity", "unit_price", "total"]
assert rows["total"].tolist() == [200.0, 250.0], "total is generated, not inserted"
assert rejected("INSERT INTO booking (customer_id, product_id, quantity, unit_price) VALUES (1, 7, 0, 10)"), "quantity of 0 must fail the CHECK"
assert rejected("INSERT INTO booking (customer_id, product_id, quantity, unit_price) VALUES (1, 7, 1, -5)"), "a negative price must fail the CHECK"
assert rejected("INSERT INTO booking (customer_id, product_id, quantity, unit_price) VALUES (1, 5, 1, 10)"), "the same customer and product twice must fail the UNIQUE"
assert rejected("INSERT INTO booking (customer_id, product_id, quantity, unit_price) VALUES (9999, 5, 1, 10)"), "a customer who does not exist must fail the FOREIGN KEY"
assert rejected("INSERT INTO booking (customer_id, product_id, quantity, unit_price, total) VALUES (2, 5, 1, 10, 99)"), "you must not be able to write to a generated column"
print("OK")

OK


## Exercise 15: Spotting the Normalization Problem  *(guide section 15)*

Show how much a denormalized column would duplicate.

For each `city`, count how many **order rows** would carry that city if it were copied onto every order, and
how many customers actually hold it once.

Columns:

- `city` — `'unknown'` where the customer has none
- `customers` — distinct customers in that city
- `order_rows` — how many orders those customers placed
- `copies_saved` — `order_rows - customers`

Order by `copies_saved` descending, then `city`.

Every unit in `copies_saved` is a place the city could be updated in one row and missed in another.

In [56]:
sql = """
SELECT
    COALESCE(c.city, 'unknown') AS city,
    COUNT(DISTINCT c.customer_id) AS customers,
    COUNT(o.order_id) AS order_rows,
    COUNT(o.order_id) - COUNT(DISTINCT c.customer_id) AS copies_saved
FROM customers c
LEFT JOIN orders o
    ON o.customer_id = c.customer_id
GROUP BY COALESCE(c.city, 'unknown')
ORDER BY copies_saved DESC, city;
"""

out = q(sql)

assert list(out.columns) == ["city", "customers", "order_rows", "copies_saved"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    ["Mumbai", 9, 71, 62],
    ["Chennai", 12, 61, 49],
    ["unknown", 5, 33, 28],
    ["Hyderabad", 5, 26, 21],
    ["Kochi", 4, 21, 17],
    ["Pune", 2, 16, 14],
    ["Bengaluru", 10, 23, 13],
    ["Kolkata", 4, 16, 12],
    ["Jaipur", 2, 13, 11],
    ["Ahmedabad", 3, 11, 8],
    ["Delhi", 3, 9, 6],
    ["Coimbatore", 1, 0, -1],
]
print("OK")

OK


## Exercise 16: Querying a Star Schema  *(guide section 16)*

The dimensional model from guide section 16 is rebuilt for you below. Write **one query** against it.

For **2024 weekdays only**, one row per `month` and `category`, with:

- `month` — from `dim_date`
- `category` — from `dim_product`
- `revenue` — `ROUND(SUM(f.revenue), 2)`
- `units` — `SUM(f.quantity)`

Keep only rows where `revenue` is above 100000. Order by `month`, then `revenue` descending. Limit 15.

Every filter should be a plain column comparison against a dimension — no `strftime` anywhere in your query.
That is the whole point of a date dimension.

In [57]:
run("""
DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_product (product_key INTEGER PRIMARY KEY, product_name TEXT, category TEXT);
CREATE TABLE dim_date (date_key TEXT PRIMARY KEY, year INTEGER, month TEXT, is_weekend INTEGER);
CREATE TABLE fact_sales (
    sale_id INTEGER PRIMARY KEY, date_key TEXT, product_key INTEGER,
    quantity INTEGER, revenue REAL);

INSERT INTO dim_product
SELECT p.product_id, p.name, c.name FROM products p JOIN categories c USING (category_id);

INSERT INTO dim_date
WITH RECURSIVE d(day) AS (
    SELECT '2023-01-01' UNION ALL SELECT date(day, '+1 day') FROM d WHERE day < '2024-12-31')
SELECT day, CAST(strftime('%Y', day) AS INTEGER), strftime('%Y-%m', day),
       CASE WHEN strftime('%w', day) IN ('0', '6') THEN 1 ELSE 0 END
FROM d;

INSERT INTO fact_sales (date_key, product_key, quantity, revenue)
SELECT o.order_date, i.product_id, i.quantity,
       ROUND(i.quantity * i.unit_price * (1 - i.discount), 2)
FROM orders o JOIN order_items i ON i.order_id = o.order_id
WHERE o.status <> 'cancelled';
""")

out = q("""
SELECT
    d.month,
    p.category,
    ROUND(SUM(f.revenue), 2) AS revenue,
    SUM(f.quantity) AS units
FROM fact_sales f
JOIN dim_date d
    ON f.date_key = d.date_key
JOIN dim_product p
    ON f.product_key = p.product_key
WHERE d.year = 2024
  AND d.is_weekend = 0
GROUP BY d.month, p.category
HAVING SUM(f.revenue) > 100000
ORDER BY d.month, revenue DESC
LIMIT 15;
""")

assert list(out.columns) == ["month", "category", "revenue", "units"]
assert len(out) == 15
assert out.round(2).values.tolist() == [
    ["2024-01", "Laptops", 236000, 2],
    ["2024-02", "Monitors", 196600, 4],
    ["2024-03", "Phones", 294000, 7],
    ["2024-03", "Monitors", 142900, 8],
    ["2024-05", "Laptops", 604000, 6],
    ["2024-05", "Monitors", 149130, 6],
    ["2024-05", "Cameras", 111400, 2],
    ["2024-06", "Monitors", 196700, 5],
    ["2024-07", "Laptops", 317600, 3],
    ["2024-08", "Phones", 256600, 9],
    ["2024-08", "Wearables", 134470, 8],
    ["2024-08", "Audio", 124750, 18],
    ["2024-09", "Laptops", 806600, 8],
    ["2024-09", "Monitors", 260200, 6],
    ["2024-09", "Phones", 159600, 4],
]
print("OK")

OK


## Exercise 17: Slowly Changing Dimension, Type 2  *(guide section 17)*

Track a change without losing history.

`dim_city` is created for you with one current row per customer for customers 1 to 4, valid from `'2023-01-01'`
to the sentinel `'9999-12-31'`.

Customer 2 moved to `'Jaipur'` on `'2024-04-01'`. Apply it as a **type 2** change:

1. Close the existing current row: set `valid_to` to the day **before** the move and `is_current` to 0.
2. Insert a new row for customer 2 with city `'Jaipur'`, `valid_from` the move date, `valid_to` the sentinel,
   and `is_current` 1.

Then set `history` to every row for customer 2 ordered by `valid_from`, and `at_the_time` to a query joining
`orders` to `dim_city` on `order_date BETWEEN valid_from AND valid_to`, returning `order_id`, `order_date` and
`city` for customer 2, ordered by `order_date`.

In [43]:
run("""
DROP TABLE IF EXISTS dim_city;
CREATE TABLE dim_city (
    city_key    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    city        TEXT,
    valid_from  TEXT NOT NULL,
    valid_to    TEXT NOT NULL DEFAULT '9999-12-31',
    is_current  INTEGER NOT NULL DEFAULT 1
);
INSERT INTO dim_city (customer_id, city, valid_from)
SELECT customer_id, city, '2023-01-01' FROM customers WHERE customer_id <= 4;
""")

run("""update dim_city set valid_to='2024-03-31' , is_current =0 where customer_id = 2; 
INSERT INTO dim_city (customer_id, city, valid_from)
SELECT customer_id, 'Jaipur', '2024-04-01'
FROM dim_city
WHERE customer_id = 2
  AND is_current = 0;
""")

history = q("""
select * from dim_city where customer_id =2 order by valid_to
""")

at_the_time = q("""SELECT o.order_id,
       o.order_date,
       d.city AS city
FROM orders o
JOIN dim_city d
  ON d.customer_id = o.customer_id
 AND o.order_date BETWEEN d.valid_from AND d.valid_to
WHERE o.customer_id = 2
ORDER BY o.order_date

""")

assert len(history) == 2, "one closed row and one current row"
assert history["is_current"].tolist() == [0, 1]
assert history["valid_to"].tolist() == ["2024-03-31", "9999-12-31"]
assert history["valid_from"].tolist() == ["2023-01-01", "2024-04-01"]
assert history["city"].tolist()[1] == "Jaipur"
assert list(at_the_time.columns) == ["order_id", "order_date", "city"]
assert len(at_the_time) == len(q("SELECT order_id FROM orders WHERE customer_id = 2")), "every order must match exactly one version -- no gaps, no overlaps"
assert set(at_the_time.loc[at_the_time["order_date"] >= "2024-04-01", "city"]) <= {"Jaipur"}
print("OK")

OK


## Exercise 18: An Idempotent Load  *(guide section 18)*

Write a loader that can be run twice without changing anything.

`daily_orders` is created for you with `date_key TEXT PRIMARY KEY`, `orders INTEGER`, `revenue REAL`.

Write a function `load_day(day)` that loads exactly one day, idempotently:

- open a transaction with `con.execute("BEGIN")`
- delete that day's row
- insert it again from `orders` joined to `order_items`, excluding cancelled orders
- commit

Then call it for `'2024-11-11'`, `'2024-11-19'` and `'2024-11-27'`, and call it for the same three days a
second time.

Pass the day with a `?` placeholder rather than formatting it into the string.

In [ ]:
run("""
DROP TABLE IF EXISTS daily_orders;
CREATE TABLE daily_orders (date_key TEXT PRIMARY KEY, orders INTEGER NOT NULL, revenue REAL NOT NULL);
""")

def load_day(day):
    # Your code here
    ...

DAYS = ["2024-11-11", "2024-11-19", "2024-11-27"]
for d in DAYS:
    load_day(d)
first_run = q("SELECT * FROM daily_orders ORDER BY date_key")

for d in DAYS:
    load_day(d)
second_run = q("SELECT * FROM daily_orders ORDER BY date_key")

assert len(first_run) == 3, "one row per day loaded"
assert first_run.equals(second_run), "running the same load twice must change nothing"
assert list(first_run.columns) == ["date_key", "orders", "revenue"]
assert first_run["date_key"].tolist() == ["2024-11-11", "2024-11-19", "2024-11-27"]
assert round(float(first_run["revenue"].sum()), 2) == round(float(q("""SELECT SUM(i.quantity * i.unit_price * (1 - i.discount)) AS r FROM orders o JOIN order_items i ON i.order_id = o.order_id WHERE o.status <> 'cancelled' AND o.order_date IN ('2024-11-11', '2024-11-19', '2024-11-27')""")["r"][0]), 2)
print("OK")

## Exercise 19: JSON Columns  *(guide section 19)*

Store documents, then query them like columns.

Create a table `api_log` with:

- `log_id INTEGER PRIMARY KEY`
- `body TEXT NOT NULL` with a `CHECK (json_valid(body))`
- `status INTEGER GENERATED ALWAYS AS (json_extract(body, '$.status')) STORED`

Insert these three rows as `body`:

```
{"status": 200, "path": "/orders", "ms": 42,  "tags": ["read", "fast"]}
{"status": 404, "path": "/nope",   "ms": 8,   "tags": ["read"]}
{"status": 500, "path": "/orders", "ms": 310, "tags": ["read", "slow", "error"]}
```

Then:

- `parsed` — `log_id`, `status`, `path` and `ms` pulled out of the JSON, ordered by `log_id`. Name the extracted
  columns `path` and `ms`.
- `tags` — one row per tag with columns `log_id` and `tag`, using `json_each`, ordered by `log_id` then `tag`.
- `bad_json` — `True` if the table refuses a row whose `body` is not valid JSON.

In [58]:
run("""
DROP TABLE IF EXISTS api_log;

CREATE TABLE api_log (
    log_id INTEGER PRIMARY KEY,
    body TEXT NOT NULL CHECK (json_valid(body)),
    status INTEGER GENERATED ALWAYS AS (
        json_extract(body, '$.status')
    ) STORED
);

INSERT INTO api_log (body) VALUES
('{"status": 200, "path": "/orders", "ms": 42, "tags": ["read", "fast"]}'),
('{"status": 404, "path": "/nope", "ms": 8, "tags": ["read"]}'),
('{"status": 500, "path": "/orders", "ms": 310, "tags": ["read", "slow", "error"]}');
""")

parsed = q("""
SELECT
    log_id,
    status,
    json_extract(body, '$.path') AS path,
    json_extract(body, '$.ms') AS ms
FROM api_log
ORDER BY log_id
""")

tags = q("""
SELECT
    api_log.log_id,
    json_each.value AS tag
FROM api_log
JOIN json_each(api_log.body, '$.tags')
ORDER BY api_log.log_id, tag

""")

try:
    run("INSERT INTO api_log (body) VALUES ('not json at all')")
    bad_json = False
except Exception:
    bad_json = True

assert list(parsed.columns) == ["log_id", "status", "path", "ms"]
assert parsed["status"].tolist() == [200, 404, 500]
assert parsed["path"].tolist() == ["/orders", "/nope", "/orders"]
assert parsed["ms"].tolist() == [42, 8, 310]
assert list(tags.columns) == ["log_id", "tag"]
assert len(tags) == 6
assert tags["tag"].tolist() == ["fast", "read", "read", "error", "read", "slow"]
assert bad_json is True, "the CHECK (json_valid(body)) should refuse it"
print("OK")

OK


## Exercise 20: Rewriting a Correlated Subquery  *(guide section 20)*

This query runs two correlated subqueries per customer:

```sql
SELECT c.customer_id,
       c.name,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS orders,
       (SELECT MAX(o.order_date) FROM orders o WHERE o.customer_id = c.customer_id) AS last_order
FROM customers c
ORDER BY c.customer_id
```

Set `fast_sql` to a rewrite that aggregates `orders` **once** in a CTE and `LEFT JOIN`s it, producing exactly
the same columns, the same order, and the same values — including `0` for customers with no orders and `NULL`
for their `last_order`.

The check compares the two results row for row, so `COALESCE` the count to 0 but leave `last_order` alone.

In [44]:
slow_sql = """
SELECT c.customer_id,
       c.name,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS orders,
       (SELECT MAX(o.order_date) FROM orders o WHERE o.customer_id = c.customer_id) AS last_order
FROM customers c
ORDER BY c.customer_id
"""

fast_sql = """
WITH order_stats AS (
    SELECT customer_id,
           COUNT(*) AS orders,
           MAX(order_date) AS last_order
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_id,
       c.name,
       COALESCE(o.orders, 0) AS orders,
       o.last_order
FROM customers c
LEFT JOIN order_stats o
       ON o.customer_id = c.customer_id
ORDER BY c.customer_id;
"""

slow = q(slow_sql)
fast = q(fast_sql)

assert list(fast.columns) == ["customer_id", "name", "orders", "last_order"]
assert len(fast) == 60
assert fast["orders"].tolist() == slow["orders"].tolist()
assert fast.fillna("-").values.tolist() == slow.fillna("-").values.tolist()
assert "WITH" in fast_sql.upper(), "aggregate once in a CTE, then join"
assert "CORRELATED" not in " ".join(plan(fast_sql)), "no correlated subquery should remain"
print("OK")

OK


## Exercise 21: Keyset Pagination  *(guide section 21)*

Replace an `OFFSET` with a cursor.

`SELECT event_id, customer_id, amount FROM events ORDER BY event_id LIMIT 10 OFFSET 150000` makes the engine
produce and throw away 150,000 rows.

1. Set `by_offset` to that query's result.
2. Set `by_keyset` to the result of a query returning **the same ten rows** by filtering on `event_id` instead
   of skipping — no `OFFSET` anywhere in it.
3. Set `offset_plan` and `keyset_plan` to the two plans.

The rows must match exactly. Think about which `event_id` the 150,001st row has, given that `event_id` runs
from 1 with no gaps.

In [45]:
offset_sql = ("SELECT event_id, customer_id, amount FROM events "
              "ORDER BY event_id LIMIT 10 OFFSET 150000")
keyset_sql = """
SELECT event_id, customer_id, amount
FROM events
WHERE event_id > 150000
ORDER BY event_id
LIMIT 10
"""

by_offset = q(offset_sql)
by_keyset = q(keyset_sql)
offset_plan = plan(offset_sql)
keyset_plan = plan(keyset_sql)

assert "OFFSET" not in keyset_sql.upper(), "the whole point is to avoid OFFSET"
assert list(by_keyset.columns) == ["event_id", "customer_id", "amount"]
assert len(by_keyset) == 10
assert by_keyset.values.tolist() == by_offset.values.tolist(), "the same ten rows"
assert by_keyset["event_id"].tolist()[0] == 150001
print("OK")

OK


## Exercise 22: Interview Problems  *(guide section 23)*

Three classics in one row.

- `second_highest_salary` — the second-highest **distinct** salary in `employees`. Use `DENSE_RANK`, so that if
  several people share it you still get one number.
- `median_salary` — the median salary, computed with the ascending/descending row-number trick, rounded to 2.
- `top_earner_count` — how many employees are on the highest salary.

Return a single row with those three columns. Use CTEs and scalar subqueries as you like.

In [53]:
sql = """
with cte as ( select * , dense_rank() over (order by salary desc ) as rn, row_number() over(order by salary) as ascn , row_number() over (order by salary desc) as desn from employees)
select (select salary from cte where rn==2)as second_highest_salary , (select salary from cte where ascn  between desn-1 and desn+1) as median_salary, (select count(*) from cte where rn=1)as top_earner_count



"""

out = q(sql)
print(out)

assert list(out.columns) == ["second_highest_salary", "median_salary", "top_earner_count"]
assert len(out) == 1
assert out.round(2).values.tolist() == [[180000, 61000, 1]]
print("OK")

   second_highest_salary  median_salary  top_earner_count
0                 180000          61000                 1
OK


## Exercise 23: Mini Project — Optimise a Slow Report End to End  *(mini project)*

A report that works and is slower than it needs to be. Make it fast, prove it, and prove it still returns the
same answer.

The report finds, for each customer, how much they spent on `'checkout'` events during 2024:

```sql
SELECT customer_id,
       COUNT(*)   AS events,
       SUM(amount) AS total
FROM events
WHERE strftime('%Y', event_date) = '2024'
  AND event_type = 'checkout'
GROUP BY customer_id
ORDER BY total DESC
```

Do three things:

1. Set `slow_plan` to its plan, after dropping any index named `ex_report`.
2. Set `fast_sql` to a rewrite where the date filter is a **half-open range** on `event_date` rather than a
   function call. Everything else stays the same, including the column names and the `ORDER BY`.
3. Create one index called **`ex_report`** that makes the rewrite index-only: it must cover the filter, the
   grouping column and the measure. Set `fast_plan` to the new plan.

The checks require the same rows and the same totals from both, a `COVERING INDEX` in the fast plan, and no
`strftime` in your rewrite.

In [59]:
slow_sql = """
SELECT customer_id,
       COUNT(*)    AS events,
       SUM(amount) AS total
FROM events
WHERE strftime('%Y', event_date) = '2024'
  AND event_type = 'checkout'
GROUP BY customer_id
ORDER BY total DESC
"""

drop_practice_indexes()
slow_plan = plan(slow_sql)

fast_sql = """
SELECT customer_id,
       COUNT(*) AS events,
       SUM(amount) AS total
FROM events
WHERE event_date >= '2024-01-01'
  AND event_date <  '2025-01-01'
  AND event_type = 'checkout'
GROUP BY customer_id
ORDER BY total DESC
"""

# Your CREATE INDEX ex_report here
run("""
CREATE INDEX ex_report
ON events(event_date, event_type, customer_id, amount)
""")

fast_plan = plan(fast_sql)
slow = q(slow_sql)
fast = q(fast_sql)
print("slow:", slow_plan)
print("fast:", fast_plan)
print(f"{timed(slow_sql):.2f} ms -> {timed(fast_sql):.2f} ms")

assert "strftime" not in fast_sql, "the date filter must be a plain range on event_date"
assert any("SCAN" in line for line in slow_plan)
assert any("COVERING INDEX" in line for line in fast_plan), "the index must hold event_type, event_date, customer_id and amount"
assert any("ex_report" in line for line in fast_plan), "name the index ex_report"
assert list(fast.columns) == ["customer_id", "events", "total"]
assert len(fast) == 21 and len(slow) == 21, "21 customers checked out during 2024"
assert fast.sort_values("customer_id").round(2).values.tolist() == slow.sort_values("customer_id").round(2).values.tolist(), "the answer must not change"
print("OK")

slow: ['SCAN events', 'USE TEMP B-TREE FOR GROUP BY', 'USE TEMP B-TREE FOR ORDER BY']
fast: ['SEARCH events USING COVERING INDEX ex_report (event_date>? AND event_date<?)', 'USE TEMP B-TREE FOR GROUP BY', 'USE TEMP B-TREE FOR ORDER BY']
41.13 ms -> 6.33 ms
OK


## Exercise 24: Mini Project — Build and Load a Star Schema  *(mini project)*

Build a small warehouse from the operational tables, load it idempotently, and query it.

1. Create four tables, dropping them first:
   - `w_dim_date(date_key TEXT PRIMARY KEY, year INTEGER, month TEXT, day_name TEXT, is_weekend INTEGER)`
   - `w_dim_product(product_key INTEGER PRIMARY KEY, product_name TEXT, category TEXT, list_price REAL)`
   - `w_dim_customer(customer_key INTEGER PRIMARY KEY, customer_name TEXT, city TEXT)`
   - `w_fact_sales(sale_id INTEGER PRIMARY KEY, date_key TEXT, customer_key INTEGER, product_key INTEGER,
     quantity INTEGER, revenue REAL)` with foreign keys to all three dimensions
2. Fill `w_dim_date` with **every day of 2023 and 2024** using a recursive CTE — 731 rows.
3. Fill the two other dimensions from `products`/`categories` and `customers`. `city` must be `'unknown'` where
   the customer has none.
4. Write `load_facts()` that deletes everything from `w_fact_sales` and reloads it from `orders` and
   `order_items`, excluding cancelled orders, all inside one transaction. Call it **twice**.
5. Set `report` to one row per `category` for 2024 weekdays only: `category`, `units`, `revenue` (rounded to 2),
   `customers` (distinct), ordered by `revenue` descending.

Every filter in step 5 must be a column on a dimension. No `strftime` in the report.

In [ ]:
run("""
-- Your DDL here
""")

run("""
-- Fill the three dimensions here
""")

def load_facts():
    # delete and reload, in one transaction
    ...

load_facts()
after_one = q("SELECT COUNT(*) AS n FROM w_fact_sales")["n"][0]
load_facts()
after_two = q("SELECT COUNT(*) AS n FROM w_fact_sales")["n"][0]

report = q("""
-- Your report here
""")

assert q("SELECT COUNT(*) AS n FROM w_dim_date")["n"][0] == 731, "every day of 2023 and 2024"
assert q("SELECT COUNT(*) AS n FROM w_dim_product")["n"][0] == 40
assert q("SELECT COUNT(*) AS n FROM w_dim_customer")["n"][0] == 60
assert q("SELECT COUNT(*) AS n FROM w_dim_customer WHERE city IS NULL")["n"][0] == 0, "COALESCE the missing cities to unknown"
assert after_one == after_two, "loading twice must not duplicate the facts"
assert after_one == q("""SELECT COUNT(*) AS n FROM order_items i JOIN orders o ON o.order_id = i.order_id WHERE o.status <> 'cancelled'""")["n"][0]
assert round(float(q("SELECT SUM(revenue) AS r FROM w_fact_sales")["r"][0]), 0) == round(float(q("""SELECT SUM(i.quantity * i.unit_price * (1 - i.discount)) AS r FROM order_items i JOIN orders o ON o.order_id = i.order_id WHERE o.status <> 'cancelled'""")["r"][0]), 0)
assert list(report.columns) == ["category", "units", "revenue", "customers"]
assert len(report) == 8, "one row per category"
assert report["revenue"].is_monotonic_decreasing
print("OK")

## Self-Review Checklist

Check whether you can do each of these **without looking at the guide**.

- [ ] Read a query plan and say whether it scans, searches, or sorts at run time.
- [ ] Explain what `ANALYZE` does and why a plan can change without the query changing.
- [ ] Add an index and prove from the plan, not the clock, that it is being used.
- [ ] Order a composite index's columns correctly, and state the leftmost prefix rule.
- [ ] Explain why a range condition ends an index's usefulness for the columns after it.
- [ ] Build a covering index and say what it saves.
- [ ] Spot a non-SARGable condition and rewrite it as a range.
- [ ] Benchmark two queries so the comparison means something.
- [ ] Say what an index costs on writes.
- [ ] Write a recursive CTE for a hierarchy, with depth and a path.
- [ ] Generate a date spine and use it to fill missing periods with zeros.
- [ ] Explain the difference between `ROWS` and `RANGE` on tied values.
- [ ] Write the gaps-and-islands query from memory.
- [ ] Say what the four ACID properties guarantee, and what an isolation level trades away.
- [ ] Use a transaction so that two statements either both happen or neither does.
- [ ] Choose between `CHECK`, `UNIQUE`, `REFERENCES` and a generated column for a given rule.
- [ ] Say what 3NF is and give a reason to break it deliberately.
- [ ] Draw a star schema and say what belongs in a fact table and what in a dimension.
- [ ] Implement a type 2 slowly changing dimension without leaving a gap or an overlap.
- [ ] Write a load that is safe to run twice, and say what makes a load unsafe.
- [ ] Extract and index a field out of a JSON column.
- [ ] Rewrite a correlated subquery as a single aggregation and join.
- [ ] Replace `OFFSET` pagination with a keyset cursor.
- [ ] Solve second-highest, median, top-N-per-group and a funnel from memory.

## What Next

You have the whole toolkit now. What is left is mileage on a real system.

- Run PostgreSQL, load a few million rows, and repeat exercises 4 to 8 there. `EXPLAIN (ANALYZE, BUFFERS)` and
  the estimated-versus-actual row counts are worth the setup on their own.
- Take the slowest query at your work. Read its plan, form one hypothesis, change one thing, re-measure. Write
  down what happened. That loop is the skill.
- Build the star schema from exercise 24 against a dataset you care about, and put a chart on top of it. A
  warehouse you loaded yourself teaches more than any explanation of one.
- Keep the two mini projects. "Here is a report I made forty times faster, here is the plan before and after"
  is one of the better things to have in an interview.